In [ ]:
# 引入依赖库以及做库的基础配置

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import get_data_file_path, load_ctg
from entry import EntryList

plt.rcParams["font.family"] = "Heiti TC"  # matplotlib 中文支持 on mac

from rich import print

In [ ]:
# 读入配置
ctg_conf = load_ctg()
print(ctg_conf)

In [ ]:
# 读入数据
data_file_path = get_data_file_path()
data = EntryList.from_csv_file(data_file_path)
data = data.filter(lambda item: item.date.year == 2025)
print(f"资产总额: {data.sum():,.2f}元")

In [ ]:
print(f"支出总额: {data.filter(lambda item: item.type == "支出").sum():,.2f}元")
print(f"收入总额: {data.filter(lambda item: item.type == "收入").sum():,.2f}元")

# 消费分布饼图
def std_pie(title: str, data: list[float], ingredients: list[str]):
    assert len(data) == len(ingredients)
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(aspect="equal"))

    def func(pct, allvals):
        absolute = int(np.round(pct / 100.0 * np.sum(allvals)))
        return f"{pct:.1f}%\n({absolute:d})"

    wedges, texts, autotexts = ax.pie(
        data,
        labels=ingredients,
        autopct=lambda pct: func(pct, data),
        textprops=dict(color="w"),
        # labeldistance=1.1,  # 控制标签距离圆心的距离
        pctdistance=0.8,  # 控制百分比标签距离圆心的距离
    )

    # 设置标签文本颜色为黑色（覆盖 textprops 的白色设置）
    for text in texts:
        text.set_fontsize(10)
        text.set_color("black")  # 修改标签文字颜色
    # ax.legend(
    #     wedges,
    #     ingredients,
    #     title=title,
    #     loc="center left",
    #     bbox_to_anchor=(1, 0, 0.5, 1),
    # )
    # plt.setp(autotexts, size=8, weight="bold")

    ax.set_title(title)

    plt.show()


# 消费分布饼图
expend_ctgs = ctg_conf["支出"].keys()
expend_ctgs_amount = {ctg: 0 for ctg in expend_ctgs}
for entry in data:
    if entry.type == "支出":
        ctg = entry.categorys[0]
        expend_ctgs_amount[ctg] += entry.amount

std_pie(
    "消费分布",
    [v for v in expend_ctgs_amount.values()],
    [k for k in expend_ctgs_amount.keys()],
)